In [ ]:
import sys
sys.path.insert(0, '..')

'd:\\202509\\实习\\game_da_prep\\endfield\\agent'

In [ ]:
from agent.llm_client import LLMClient
from agent.sentiment import classify

In [2]:
import pandas as pd
import os
from tqdm import tqdm
import time

In [3]:
client = LLMClient()

In [9]:
df = pd.read_csv("D:\\202509\\实习\\game_da_prep\\endfield\data\\taptap_reviews_full.csv")

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\lenovo\AppData\Local\Temp\ipykernel_9144\1078740250.py:1: SyntaxWarning: invalid escape sequence '\d'
  df = pd.read_csv("D:\\202509\\实习\\game_da_prep\\endfield\data\\taptap_reviews_full.csv")


In [ ]:
sample_text = df.sample(1, random_state=42)['content'].iloc[0]
print(f"输入评论:\n{sample_text}\n")
result = classify(sample_text, client)
print(f"返回结果:\n{result}")

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\lenovo\AppData\Local\Temp\ipykernel_9144\3383087831.py:1: SyntaxWarning: invalid escape sequence '\D'
  df = pd.read_csv("\D:\\202509\\实习\\game_da_prep\\endfield\data\\taptap_reviews_full.csv")
C:\Users\lenovo\AppData\Local\Temp\ipykernel_9144\3383087831.py:1: SyntaxWarning: invalid escape sequence '\D'
  df = pd.read_csv("\D:\\202509\\实习\\game_da_prep\\endfield\data\\taptap_reviews_full.csv")


OSError: [Errno 22] Invalid argument: '\\D:\\202509\\实习\\game_da_prep\\endfield\\data\\taptap_reviews_full.csv'

In [ ]:
os.makedirs("D:\\202509\\实习\\game_da_prep\\endfield\\eval", exist_ok=True)

df = pd.read_csv("D:\\202509\\实习\\game_da_prep\\endfield\\data\\taptap_reviews_full.csv")

# 抽 100 条固定样本
eval_v1 = df.sample(100, random_state=42).reset_index(drop=True)

# 只保留必要字段
eval_v1 = eval_v1[['review_id', 'score', 'content', 'created_date']]

# 加空列供后续填充
eval_v1['label_true'] = ''       # 人工标注列
eval_v1['pred_sentiment'] = ''   # LLM 预测列
eval_v1['pred_confidence'] = 0.0

# 存 CSV
eval_v1.to_csv("D:\\202509\\实习\\game_da_prep\\endfield\\eval\\eval_v1.csv", index=False, encoding='utf-8-sig')
print(f"✅ 已存 100 条")
print(f"score 分布:\n{eval_v1['score'].value_counts().sort_index()}")

In [ ]:
client = LLMClient()

eval_df = pd.read_csv("D:\\202509\\实习\\game_da_prep\\endfield\\eval\\eval_v1.csv")

for i, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    try:
        result = classify(row['content'], client)
        eval_df.at[i, 'pred_sentiment'] = result.get('sentiment', 'error')
        eval_df.at[i, 'pred_confidence'] = result.get('confidence', 0)
    except Exception as e:
        print(f"[{i}] err: {e}")
        eval_df.at[i, 'pred_sentiment'] = 'error'
    time.sleep(0.3)  # 保守限流

# 覆盖存回
eval_df.to_csv("D:\\202509\\实习\\game_da_prep\\endfield\\eval\\eval_v1.csv", index=False, encoding='utf-8-sig')
print(f"✅ 完成 {len(eval_df)} 条预测")
print(eval_df['pred_sentiment'].value_counts())

In [5]:
eval_df = pd.read_csv("D:\\202509\\实习\\game_da_prep\\endfield\\eval\\eval_v1.csv")

# 过滤有效数据（去掉 error 和未标注）
valid = eval_df[
    (eval_df['label_true'].isin(['positive', 'negative', 'neutral'])) &
    (eval_df['pred_sentiment'].isin(['positive', 'negative', 'neutral']))
]

print(f"有效样本: {len(valid)}/{len(eval_df)}")

# 总体准确率
accuracy = (valid['pred_sentiment'] == valid['label_true']).mean()
print(f"\n★ 总体准确率: {accuracy:.1%}")

# 分类别准确率
print("\n分类别精确率:")
for label in ['positive', 'negative', 'neutral']:
    subset = valid[valid['label_true'] == label]
    if len(subset) > 0:
        acc = (subset['pred_sentiment'] == label).mean()
        print(f"  {label}: {acc:.1%} (n={len(subset)})")

# 混淆矩阵
from sklearn.metrics import confusion_matrix, classification_report
labels = ['positive', 'negative', 'neutral']
cm = confusion_matrix(valid['label_true'], valid['pred_sentiment'], labels=labels)
cm_df = pd.DataFrame(cm, index=[f'true_{l}' for l in labels], 
                     columns=[f'pred_{l}' for l in labels])
print(f"\n混淆矩阵:\n{cm_df}")

print(f"\n详细分类报告:\n{classification_report(valid['label_true'], valid['pred_sentiment'], labels=labels)}")

有效样本: 100/100

★ 总体准确率: 73.0%

分类别精确率:
  positive: 91.5% (n=59)
  negative: 94.7% (n=19)
  neutral: 4.5% (n=22)

混淆矩阵:
               pred_positive  pred_negative  pred_neutral
true_positive             54              5             0
true_negative              1             18             0
true_neutral               7             14             1

详细分类报告:
              precision    recall  f1-score   support

    positive       0.87      0.92      0.89        59
    negative       0.49      0.95      0.64        19
     neutral       1.00      0.05      0.09        22

    accuracy                           0.73       100
   macro avg       0.79      0.64      0.54       100
weighted avg       0.83      0.73      0.67       100



In [6]:
errors = valid[valid['pred_sentiment'] != valid['label_true']].copy()
bad_case = errors.iloc[0]
print(f"评论: {bad_case['content']}")
print(f"人工: {bad_case['label_true']}")
print(f"旧预测: {bad_case['pred_sentiment']}")
print(f"新预测: {classify(bad_case['content'], client)}")

评论: 看了一圈，评论区大多是对这个游戏期望值过高，希望体验感比明日方舟更好，而忽略了一个ip再次消费后必然产生的质量降低。说实话在二游里质量中等而平庸，但不是地板，也不是那么多谣言里的样子。不过想入坑的一定要深思熟虑，这游戏本身适合周末或假期的大块时间，而不是碎片化；另外评论区的玩家群体里骂完接着玩和不玩就骂的也不少，一切体验都要跟着自己感官走，不要跟风。当然如果有难绷脑残儿上来就贴孝标签的话，我也没招，毕竟您都高强度刷评论区了您啥成分有目共睹。<br />我真正的评价是背景音乐好听、武陵地图设计不错，但可玩性不够，配音一般，角色形象太平面，剧情也就市面上一般水平（要是你说一坨的话我不反对，毕竟市面上的绝大部分二游也都就是一坨），玩法什么的跟其他游戏里大差不差，战斗系统严重怀疑是原神和鸣潮那一套。总而言之，太过平庸，不至于一坨，也不是啥神作，就是……没啥亮点就是它最大的缺点。它没那么差，但是严重和前作的质量脱节，简而说就是没填满大部分人心里的期望，所以很多人，很多大v大up也会，来通过骂和差评的方式来宣泄情绪。对了，开服福利没那么少，虽然也没那么多，但个人感觉跟其他游戏没差多少，无非就是保底机制不行，但这游戏属于那种配个队就能玩的那种，白给的管理和小陈的强度已经够乱杀了，有没有up关系貌似不大。<br />最后希望正在制作的武陵主线能挽回点口碑，要不过到时候我还得回来改评价降评分，麻烦。ps：玩的官服，评价纯属tap上刷到了顺手评的
人工: neutral
旧预测: negative
新预测: {'sentiment': 'neutral', 'confidence': 0.7}


In [14]:
# 关键: 避开前100条 (index 0-99 in sampled), 用不同 random_state
already_used_ids = eval_df['review_id'].tolist()
remaining = df[~df['review_id'].isin(already_used_ids)]
eval_v2 = remaining.sample(50, random_state=12345).reset_index(drop=True)
eval_v2 = eval_v2[['review_id', 'score', 'content', 'created_date']]
eval_v2['label_true'] = ''
eval_v2['pred_sentiment'] = ''
eval_v2['pred_confidence'] = 0.0
eval_v2.to_csv("D:\\202509\\实习\\game_da_prep\\endfield\\eval\\eval_v2.csv", index=False, encoding='utf-8-sig')

In [15]:
for i, row in tqdm(eval_v2.iterrows(), total=50):
    try:
        result = classify(row['content'], client)
        eval_v2.at[i, 'pred_sentiment'] = result.get('sentiment', 'error')
        eval_v2.at[i, 'pred_confidence'] = result.get('confidence', 0)
    except Exception as e:
        eval_v2.at[i, 'pred_sentiment'] = 'error'
    time.sleep(0.3)

eval_v2.to_csv("D:\\202509\\实习\\game_da_prep\\endfield\\eval\\eval_v2.csv", index=False, encoding='utf-8-sig')

100%|██████████| 50/50 [01:08<00:00,  1.37s/it]


In [17]:
eval_v2 = pd.read_csv("D:\\202509\\实习\\game_da_prep\\endfield\\eval\\eval_v2.csv")
valid_v2 = eval_v2[
    eval_v2['label_true'].isin(['positive', 'negative', 'neutral']) &
    eval_v2['pred_sentiment'].isin(['positive', 'negative', 'neutral'])
]
accuracy_v2 = (valid_v2['pred_sentiment'] == valid_v2['label_true']).mean()
print(f"第二轮准确率: {accuracy_v2:.1%}")

第二轮准确率: 86.0%
